# Cassini CDA Dust Analyzer: Peaks-Focused Ablation Study with QI Amplitude Filtering

This notebook conducts a specialized ablation study focusing on the **3-P** dust particle class with **QI amplitude filtering** and **peaks-focused spectral prompts** (excluding y-axis baseline level/shift from the classification criteria).

### Study Configurations:
- **QI Amplitude Filter**: $10\text{ fC} \le Q_i \le 2\text{ pC}$ (i.e., `1e-14 <= qi_ampl <= 2e-12` C).
- **Peaks-Focused Prompts**: Updated system instructions and contrastive guidelines to ignore y-axis baseline levels and vertical baseline shifts, focusing exclusively on peak positions, peak spacing, peak shape, and peak counts.
- **No Self-Correction**: This run evaluates the first-pass multiclass VLM performance directly without running the Stage 2 self-correction loop.
- **Train Splitting ($k$-shots)**:
  - **$k = 4$ shots** for Class **3-P** (to avoid oversampling the rare class).
  - **$k = 16$ shots** for standard classes (Noise, 1, 2, 3, 4, 5, 5-Na).
- **Test Splitting**: Capped at **100 samples per class** (or all available if a class has fewer than 100 samples, like Class 3-P which has 5 test samples).
- **Outputs**: Saves results as CSV and Parquet inside `Notebooks/study_results_xx_3p_peaks_focused/` along with confusion matrices (CSV and PNG) and metrics JSON.


## Step 1: Imports and Configurations


In [ ]:
import os
import io
import json
import base64
import asyncio
import time
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime
from google import genai
from google.cloud import storage
from google.genai import types
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.signal import savgol_filter
import huggingface_hub
import tqdm
import gc

warnings.filterwarnings("ignore")
import logging
logging.getLogger("google_genai").setLevel(logging.WARNING)

from dotenv import load_dotenv
try:
    dotenv_path = os.path.join(os.path.dirname(__file__), '../.env')
    project_root = os.path.join(os.path.dirname(__file__), '..')
except NameError:
    dotenv_path = "../.env"
    project_root = ".."

load_dotenv(dotenv_path=dotenv_path)

import sys
sys.path.append(os.path.abspath(project_root))

from cda_dust_agent.config import Config
from cda_dust_agent.tools.utils import generate_spectrum_image_bytes, parse_response


## Step 2: Load Peaks-Focused Prompts & Streaming Helper Functions


In [ ]:
SYSTEM_INSTRUCTION_TEXT = """You are Antigravity, an expert Cosmic Dust Spectroscopist analyzing Cassini Cosmic Dust Analyzer (CDA) mass spectra.
Your task is to classify target mass spectra into one of the following classes:
- Class 1 (Pure water ice)
- Class 2 (Organic-rich water ice)
- Class 3 (Organics)
- Class 4 (Minerals)
- Class 5 (Sputtered alkali core / Bimodal extreme)
- Class 5-Na (Sodium-rich bipartite)
- Class 3-P (Burst-and-trail signature)
- Noise (Digitizer/instrumental noise)

IMPORTANT CLASSIFICATION CRITERIA:
1. The vertical level of the baseline or any shift along the y-axis is NOT a valid classification criterion and must be ignored.
2. Focus EXCLUSIVELY on the peak positions (x-axis channels), the spacing/distance between peaks, the number of peaks, and their shapes (e.g. sharp spikes vs broad envelopes).

Spectral Guideline Descriptions:
- Class 1: Repeating sequence of sharp, regularly spaced hydronium cluster peaks ($H_3O^+(H_2O)_n$) at mass-anchored positions ~19, 37, 55, 73... (index channels ~85, ~120, ~146, ~169, ~189), starting with an early global maximum.
- Class 2: Same repeating sequence of hydronium cluster peaks at index ~85, ~120, ~146, ~169, ~189, but with extra intermediate peaks between them (valley-filling peaks representing organic/volatile contamination).
- Class 3: Multiple broad asymmetric peak clusters (shark-fin profiles) representing carbon/organic groups, typically peaking near indices ~220, ~360, and ~530.
- Class 4: Narrow, needle-sharp atomic cation peaks (Mg+, Si+, Fe+) at early index channels (e.g. ~10, ~135, ~150), followed by broad mineral/silicate peak complexes peaking near index ~320-350 and ~450. Focus on the absence of repeating water-ice peak spacing.
- Class 5: An overwhelmingly intense, sharp early peak (often double-peaked near indices ~140 and ~210) followed by a quiet, flat barcode-like low-amplitude signal devoid of major peaks.
- Class 5-Na: A singular, extremely sharp sodium peak near index ~130, followed by a wide, flat flat-topped detector-saturation peak envelope peaking near index ~350.
- Class 3-P: A massive primary peak complex (global maximum near index ~200-220) and a distinct secondary peak near index ~330-345. Focus on the combination of these two major peak positions and their unique envelopes.
- Noise: High-frequency grass-like barcode peaks without any chemically coherent peak clusters or sharp atomic spikes."""

GENERAL_PROFILES = {
    "Class 1": "Repeating sequence of sharp, regularly spaced hydronium cluster peaks starting with an early global maximum.",
    "Class 2": "Repeating sequence of hydronium cluster peaks with extra intermediate peaks between them.",
    "Class 3": "Multiple broad asymmetric peak clusters representing carbon/organic groups, typically peaking near indices 220, 360, and 530.",
    "Class 4": "Narrow, needle-sharp atomic cation peaks at early channels, followed by broad mineral/silicate peak complexes peaking near index 320-350 and 450.",
    "Class 5": "An overwhelmingly intense, sharp early peak followed by a quiet, flat barcode-like low-amplitude signal devoid of major peaks.",
    "Class 5-Na": "A singular, extremely sharp sodium peak near index 130, followed by a wide, flat flat-topped detector-saturation peak envelope peaking near index 350.",
    "Class 3-P": "A massive primary peak complex (global maximum near index 200-220) and a distinct secondary peak near index 330-345.",
    "Noise": "High-frequency grass-like barcode peaks without any chemically coherent peak clusters or sharp atomic spikes."
}

CLASSIFICATION_USER_PROMPT = """Review the provided mass spectrum and classify it into one of the known classes.
Focus only on peak positions, spacing/distances between peaks, peak shapes, and peak counts. Ignore any baseline shift on the y-axis.

Format your output strictly as a JSON object with keys:
{
    "id": "<the provided sclk id>",
    "class": "<the predicted class label>",
    "explanation": "<a brief 1-2 sentence explanation of why it belongs to this class based on visual peak features>",
    "k": <the value of k shots used>
}
"""

ANNOTATION_USER_PROMPT = """This is a time-of-flight mass spectrum for a particle belonging to the known class '{label}'.
Please provide a brief, 1-2 sentence description of the key visual features (peaks, spacing, positions) that characterize this spectrum as '{label}'. Do not output JSON, just the text description."""

CONTRASTIVE_ANNOTATION_USER_PROMPT = """You are an expert Cosmic Dust Spectroscopist.
We want you to write a brief, 1-2 sentence description of the key visual features of the Target Spectrum (labeled as Class '{label}') to be used as a reference example for class '{label}'.

To help you write a contrastive explanation that clearly distinguishes '{label}' from all other classes, we have provided the Target Spectrum (Image A) alongside reference spectra from the other classes.

Please review all the provided images:
- Image A (Target): This is the spectrum of '{label}' you must describe.
{reference_descriptions}

Key Spectral Guidelines to remember:
- The baseline level and its vertical shift on the y-axis are NOT a valid classification criterion and must be ignored. Focus ONLY on peaks, their positions, spacing, number, and shapes.
- Class 1: Repeating sequence of sharp, regularly spaced hydronium cluster peaks ($H_3O^+(H_2O)_n$) at mass-anchored positions ~19, 37, 55, 73... (index channels ~85, ~120, ~146, ~169, ~189), starting with an early global maximum.
- Class 2: Same repeating sequence of hydronium cluster peaks at index ~85, ~120, ~146, ~169, ~189, but with extra intermediate peaks between them (valley-filling peaks representing organic/volatile contamination).
- Class 3: Multiple broad asymmetric peak clusters (shark-fin profiles) representing carbon/organic groups, typically peaking near indices ~220, ~360, and ~530.
- Class 4: Narrow, needle-sharp atomic cation peaks (Mg+, Si+, Fe+) at early index channels (e.g. ~10, ~135, ~150), followed by broad mineral/silicate peak complexes peaking near index ~320-350 and ~450. Focus on the absence of repeating water-ice peak spacing.
- Class 5: An overwhelmingly intense, sharp early peak (often double-peaked near indices ~140 and ~210) followed by a quiet, flat barcode-like low-amplitude signal devoid of major peaks.
- Class 5-Na: A singular, extremely sharp sodium peak near index ~130, followed by a wide, flat flat-topped detector-saturation peak envelope peaking near index ~350.
- Class 3-P: A massive primary peak complex (global maximum near index ~200-220) and a distinct secondary peak near index ~330-345. Focus on the combination of these two major peak positions and their unique envelopes.
- Noise: High-frequency grass-like barcode peaks without any chemically coherent peak clusters or sharp atomic spikes.

Write a 1-2 sentence description of the visual features of Image A (Target) that characterize it as '{label}', highlighting specific details (such as peak spacing trend, peak positions, number, or shape) that help distinguish it from the other classes. Do not output JSON, just return the text description."""

def extract_prediction_from_line(line):
    idx = line.rfind('"text":"{')
    if idx == -1:
        idx = line.rfind('"text":"{\\n')
    if idx == -1:
        return None
        
    start_val_idx = idx + 7  # index of the double quote of the string value
    
    end_idx = -1
    pos = start_val_idx + 1
    while pos < len(line):
        if line[pos] == '"':
            # Count backslashes before it
            bs_count = 0
            k = pos - 1
            while k >= start_val_idx and line[k] == '\\':
                bs_count += 1
                k -= 1
            if bs_count % 2 == 0:
                end_idx = pos
                break
        pos += 1
        
    if end_idx == -1:
        return None
        
    escaped_json_str = line[start_val_idx:end_idx+1]
    try:
        json_str = json.loads(escaped_json_str)
        return json.loads(json_str)
    except Exception:
        return None


## Step 3: Fetching and Preprocessing Data (With pre-split QI filtering)


In [ ]:
def download_and_preprocess_data():
    print("Downloading train dataset from Hugging Face...")
    REPO_ID = "CosmicDustGroup/cassini-cda-spectra"
    FILENAME_TRAIN = "data/lvl2/cda_qm_spectra_pre2008277_train_lvl2.parquet"
    
    file_train_path = huggingface_hub.hf_hub_download(repo_id=REPO_ID, filename=FILENAME_TRAIN, repo_type="dataset")
    print(f"Downloaded train dataset to: {file_train_path}")
    
    df = pd.read_parquet(file_train_path)
    print(f"Loaded {len(df)} raw spectra.")
    
    # Filter 1018 length spectra
    df = df[df['spectrum'].apply(len) == 1018].copy()
    print(f"Filtered to {len(df)} spectra of length 1018.")
    
    # Crop spectrum to index 10 to 640
    def crop_spectrum(spectrum):
        return spectrum[10:641]
    df['spectrum'] = df['spectrum'].apply(crop_spectrum)
    
    # Map labels
    def map_label(x):
        if not isinstance(x, str):
            return "?"
        if x in ["?", "X", "2-X"]:
            return "?"
        if x == "3-P":
            return "3-P"
        if x.startswith("3-") or x == "3":
            return "3"
        return x
        
    df['class'] = df['class'].apply(map_label)
    df = df[df['class'] != '?'].copy()
    print(f"Remaining active spectra after mapping: {len(df)}")
    
    # Apply QI Amplitude Filtering before train/test splitting
    # 10 fC to 2 pC
    lower_limit = 10 * 1e-15  # 1e-14 C
    upper_limit = 2 * 1e-12   # 2e-12 C
    
    df = df[(df['qi_ampl'] >= lower_limit) & (df['qi_ampl'] <= upper_limit)].copy()
    print(f"Applied QI filtering (10 fC <= QI <= 2 pC). Remaining spectra: {len(df)}")
    print("Class distribution after filtering:")
    print(df['class'].value_counts())
    
    # Scaling and smoothing
    def qm_scaling_savgol(spectrum):
        spectrum = np.array(spectrum, dtype=float)
        # Apply log10
        log_spec = np.log10(spectrum + np.abs(np.min(spectrum)))
        
        # Replace negative infinity/NaN with minimum finite value
        finite_mask = np.isfinite(log_spec)
        if np.any(finite_mask):
            min_finite_val = np.min(log_spec[finite_mask])
        else:
            min_finite_val = 0
        log_spec = np.nan_to_num(log_spec, neginf=min_finite_val, nan=min_finite_val)
        
        # Min-max scaling
        spec_min = np.min(log_spec)
        spec_max = np.max(log_spec)
        range_val = spec_max - spec_min
        if range_val > 0:
            scaled_spectrum = (log_spec - spec_min) / range_val
        else:
            scaled_spectrum = np.zeros_like(log_spec)
            
        return savgol_filter(scaled_spectrum, 5, 3)
        
    df['spectrum'] = df['spectrum'].apply(qm_scaling_savgol)
    print("Scaling and smoothing complete.")
    
    return df

df = download_and_preprocess_data()


## Step 4: Partitioning Data into Train and Test Pools (With max training limit and 100 max cap test limit per class)


In [ ]:
def split_train_test(df, max_train_map=None, test_size_per_class=None):
    if max_train_map is None:
        max_train_map = {
            '3-P': 4,
            '5': 16,
            '5-Na': 16
        }
        
    train_sclks = {}
    sampled_rows = []
    
    classes = ['Noise', '1', '2', '3', '4', '5', '5-Na', '3-P']
    
    for cls in classes:
        cls_df = df[df['class'] == cls]
        # Train limit based on map, default is 16
        n_train = max_train_map.get(cls, 16)
        n_train = min(n_train, len(cls_df))
        
        cls_train = cls_df.sample(n=n_train, random_state=123)
        train_sclks[cls] = cls_train['sclk'].tolist()
        sampled_rows.append(cls_train)
        
    train_df = pd.concat(sampled_rows, ignore_index=True)
    all_train_sclks = [sclk for sclks in train_sclks.values() for sclk in sclks]
    
    test_pool = df[~df['sclk'].isin(all_train_sclks)].copy()
    
    test_dfs = []
    for cls in classes:
        cls_test_pool = test_pool[test_pool['class'] == cls]
        if test_size_per_class is not None:
            n_samples = min(test_size_per_class, len(cls_test_pool))
        else:
            n_samples = len(cls_test_pool)
            
        if n_samples > 0:
            cls_test = cls_test_pool.sample(n=n_samples, random_state=42)
            test_dfs.append(cls_test)
            print(f"Included class '{cls}' in test set with {len(cls_test)} samples.")
        
    test_df = pd.concat(test_dfs, ignore_index=True)
    print(f"Train samples: {len(train_df)} (classes breakdown: { {k: len(v) for k, v in train_sclks.items()} })")
    print(f"Test samples: {len(test_df)}")
    
    return train_df, test_df, train_sclks

# Split with a maximum cap of 100 samples per class
train_df, test_df, train_sclks = split_train_test(df, test_size_per_class=100)


## Step 5: Generating VLM Explanations Concurrently


In [ ]:
async def generate_explanation(client, model_id, spectrum_array, label, sclk, semaphore, references=None):
    async with semaphore:
        def format_class_label(cls):
            cls_str = str(cls)
            if cls_str.lower().startswith("class"):
                return cls_str
            return f"Class {cls_str}" if cls_str.lower() != 'noise' else "Noise"

        target_label = format_class_label(label)
        img_bytes = generate_spectrum_image_bytes(np.array(spectrum_array), title=f"{target_label} Sample {sclk}")
        
        if references:
            ref_descriptions = ""
            for idx, (other_cls, _) in enumerate(references):
                char_code = chr(66 + idx) # B, C, D, E...
                ref_descriptions += f"- Image {char_code}: Reference Spectrum of '{format_class_label(other_cls)}'\\n"
                
            prompt = CONTRASTIVE_ANNOTATION_USER_PROMPT.format(
                label=target_label,
                reference_descriptions=ref_descriptions.strip()
            )
            
            parts = [
                types.Part.from_text(text=prompt),
                types.Part.from_bytes(data=img_bytes, mime_type="image/png") # Image A
            ]
            for _, other_img_bytes in references:
                parts.append(types.Part.from_bytes(data=other_img_bytes, mime_type="image/png"))
                
            contents = [types.Content(role="user", parts=parts)]
        else:
            prompt = ANNOTATION_USER_PROMPT.format(label=label)
            contents = [
                types.Content(role="user", parts=[
                    types.Part.from_text(text=prompt),
                    types.Part.from_bytes(data=img_bytes, mime_type="image/png")
                ])
            ]
        
        retries = 3
        for attempt in range(retries):
            try:
                response = await client.aio.models.generate_content(
                    model=model_id,
                    contents=contents,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_INSTRUCTION_TEXT
                    )
                )
                return {
                    "label": label,
                    "sclk": sclk,
                    "image": img_bytes,
                    "explanation": response.text.strip(),
                    "image_base64": base64.b64encode(img_bytes).decode('utf-8')
                }
            except Exception as e:
                print(f"Failed to generate explanation for {label} (sclk: {sclk}): {e}. Retrying {attempt+1}/{retries}...")
                await asyncio.sleep(2 ** attempt)
        return None

async def generate_explanations_pool(client, model_id, train_df, train_sclks):
    print("Pre-rendering and caching training spectrum images...")
    image_cache = {}
    for _, row in train_df.iterrows():
        sclk = row['sclk']
        cls = row['class']
        label = f"Class {cls}" if str(cls).lower() != 'noise' else "Noise"
        img_bytes = generate_spectrum_image_bytes(np.array(row['spectrum']), title=f"{label} Reference Sample {sclk}")
        image_cache[sclk] = img_bytes
        
    api_semaphore = asyncio.Semaphore(10)
    explanation_tasks = []
    
    import random
    random.seed(123)
    
    print(f"Generating explanations for {len(train_df)} training samples concurrently with dynamic contrastive references...")
    for _, row in train_df.iterrows():
        target_sclk = row['sclk']
        target_cls = row['class']
        
        references = []
        for other_cls, other_sclks in train_sclks.items():
            if other_cls != target_cls:
                other_sclk = random.choice(other_sclks)
                references.append((other_cls, image_cache[other_sclk]))
                
        task = generate_explanation(client, model_id, row['spectrum'], row['class'], row['sclk'], api_semaphore, references=references)
        explanation_tasks.append(task)
            
    results = await asyncio.gather(*explanation_tasks)
    explanations_pool = [r for r in results if r is not None]
    print(f"Successfully generated {len(explanations_pool)} explanations.")
    return explanations_pool

configs = Config()
client = genai.Client()
model_id = configs.agent_settings.model
explanations_pool = await generate_explanations_pool(client, model_id, train_df, train_sclks)


## Step 6: Create Request JSONL File (Single Job Request)


In [ ]:
def create_ablation_batch_input_file(test_df, explanations_pool, k_value, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "ablation_requests_3p.jsonl")
    print(f"Generating single batch request file: {output_file}...")
    
    class_explanations = {}
    for ex in explanations_pool:
        cls = ex['label']
        if cls not in class_explanations:
            class_explanations[cls] = []
        class_explanations[cls].append(ex)
        
    print(f"Pre-rendering and caching target spectrum images for {len(test_df)} samples...")
    test_image_cache = {}
    fig, ax = plt.subplots(figsize=(12, 6), dpi=60)
    
    for idx, row in test_df.iterrows():
        sclk = row['sclk']
        if sclk not in test_image_cache:
            ax.clear()
            ax.plot(row['spectrum'], color='black', linewidth=1.5)
            ax.set_title(f"Sample {sclk}")
            ax.set_ylim(-0.02, 1.02)
            ax.set_xlim(0, 630)
            ax.grid(True)
            
            buf = io.BytesIO()
            fig.savefig(buf, format='png')
            buf.seek(0)
            img_bytes = buf.getvalue()
            img_b64 = base64.b64encode(img_bytes).decode('utf-8')
            test_image_cache[sclk] = img_b64
            
        if (idx + 1) % 100 == 0 or (idx + 1) == len(test_df):
            print(f"  -> Pre-rendered {idx + 1}/{len(test_df)} images...")
            
    plt.close(fig)
    gc.collect()
    
    k_shot_examples = []
    for cls, exs in class_explanations.items():
        limit = 4 if cls == "3-P" else 16
        k_shot_examples.extend(exs[:limit])
        
    used_sclk_ids = [ex['sclk'] for ex in k_shot_examples]
    
    few_shot_parts = [{"text": CLASSIFICATION_USER_PROMPT}]
    if k_shot_examples:
        few_shot_parts.append({"text": "Here are reference examples:"})
        for ex in k_shot_examples:
            label_key = ex['label']
            gen_profile = GENERAL_PROFILES.get(label_key, '')
            class_label = f"Class {label_key}" if str(label_key).lower() != 'noise' else "Noise"
            few_shot_parts.append({"text": f"Example: {class_label}\\n- General Profile: {gen_profile}\\n- Specific Sample Features: {ex['explanation']}"})
            few_shot_parts.append({"inline_data": {"mime_type": "image/png", "data": ex['image_base64']}})
        few_shot_parts.append({"text": "Now, analyze the following spectrum:"})
        
    requests = []
    for _, row in test_df.iterrows():
        if row['sclk'] in used_sclk_ids:
            continue
            
        img_b64 = test_image_cache[row['sclk']]
        
        current_parts = few_shot_parts.copy()
        current_parts.append({"inline_data": {"mime_type": "image/png", "data": img_b64}})
        current_parts.append({"text": f"Sample ID (sclk): {row['sclk']} | k: {k_value}"})
        
        request = {
            "request": {
                "contents": [
                    {
                        "role": "user",
                        "parts": current_parts
                    }
                ],
                "systemInstruction": {
                    "parts": [{"text": SYSTEM_INSTRUCTION_TEXT}]
                },
                "generationConfig": {
                    "responseMimeType": "application/json",
                    "responseSchema": {
                        "type": "OBJECT",
                        "properties": {
                            "id": {"type": "STRING", "description": "The ID of the run (sclk)"},
                            "class": {"type": "STRING", "description": "The predicted class label"},
                            "explanation": {"type": "STRING", "description": "Explanation for the prediction"},
                            "k": {"type": "INTEGER", "description": "The value of k shots used"}
                        },
                        "required": ["id", "class", "explanation", "k"]
                    }
                }
            }
        }
        requests.append(json.dumps(request))
        
    with open(output_file, 'w') as f:
        for req in requests:
            f.write(req + '\n')
            
    print(f"Generated JSONL file with {len(requests)} requests at {output_file}")
    del test_image_cache
    gc.collect()
    
    return output_file

# Absolute path output directory
output_dir = os.path.abspath(os.path.join(project_root, "Notebooks/study_results_xx_3p_peaks_focused"))
k_value = 4
batch_file = create_ablation_batch_input_file(test_df, explanations_pool, k_value, output_dir)


## Step 7: Submit, Poll, Stream-Download and Parse Prediction results


In [ ]:
def submit_and_poll_batch_job(configs, local_jsonl_path):
    project_id = configs.agent_settings.project_id
    bucket_name = configs.agent_settings.bucket_name
    
    filename = os.path.basename(local_jsonl_path)
    print(f"Uploading {local_jsonl_path} to GCS bucket: {bucket_name}")
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.bucket(bucket_name.replace("gs://", ""))
    
    gcs_input_path = f"input/{filename}"
    blob = bucket.blob(gcs_input_path)
    
    with open(local_jsonl_path, 'rb') as f:
        blob.chunk_size = 10 * 1024 * 1024
        blob.upload_from_file(f, content_type="application/json", timeout=600)
        
    gcs_source = f"gs://{bucket.name}/{gcs_input_path}"
    print(f"Uploaded to {gcs_source}")
    
    access_token = os.popen("gcloud auth application-default print-access-token").read().strip()
    job_display_name = f"cda-3p-ablation-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    
    global_batch_req = {
        "displayName": job_display_name,
        "model": f"publishers/google/models/{configs.agent_settings.model}",
        "inputConfig": {
            "instancesFormat": "jsonl",
            "gcsSource": {"uris": [gcs_source]}
        },
        "outputConfig": {
            "predictionsFormat": "jsonl",
            "gcsDestination": {"outputUriPrefix": f"gs://{bucket.name}/output"}
        }
    }
    
    req_file = f"batch_request_3p_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(req_file, "w") as f:
        json.dump(global_batch_req, f)
        
    curl_command = [
        "curl", "-s", "-X", "POST",
        f"https://aiplatform.googleapis.com/v1/projects/{project_id}/locations/global/batchPredictionJobs",
        "-H", f"Authorization: Bearer {access_token}",
        "-H", "Content-Type: application/json; charset=utf-8",
        "-d", f"@{req_file}"
    ]
    
    print("Submitting Batch Job to Vertex AI...")
    result = subprocess.run(curl_command, capture_output=True, text=True)
    
    if os.path.exists(req_file):
        os.remove(req_file)
        
    if result.returncode == 0 and "name" in result.stdout:
        response_json = json.loads(result.stdout)
        job_name = response_json.get("name")
        print(f"Job Submitted! Name: {job_name}")
    else:
        print(f"Error submitting job: {result.stderr}\\nResponse: {result.stdout}")
        raise RuntimeError("Failed to submit Batch Prediction Job.")
        
    check_url = f"https://aiplatform.googleapis.com/v1/{job_name}"
    print("Polling job status every 30 seconds...")
    while True:
        access_token = os.popen("gcloud auth application-default print-access-token").read().strip()
        check_cmd = [
            "curl", "-s", "-X", "GET",
            check_url,
            "-H", f"Authorization: Bearer {access_token}"
        ]
        check_res = subprocess.run(check_cmd, capture_output=True, text=True)
        if check_res.returncode != 0:
            print(f"Error checking status: {check_res.stderr}")
            time.sleep(30)
            continue
            
        status_data = json.loads(check_res.stdout)
        state = status_data.get("state", "UNKNOWN")
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Job State: {state}")
        
        if state == "JOB_STATE_SUCCEEDED":
            print("Job Completed Successfully!")
            return status_data
        elif state in ["JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_PAUSED"]:
            err_msg = status_data.get("error", "No error details.")
            raise RuntimeError(f"Job ended with state {state}. Details: {err_msg}")
            
        time.sleep(30)

def download_and_parse_batch_results_streaming(configs, job_status_data, test_df, output_parquet_path):
    project_id = configs.agent_settings.project_id
    bucket_name = configs.agent_settings.bucket_name
    
    gcs_output_dir = job_status_data.get("outputInfo", {}).get("gcsOutputDirectory")
    if not gcs_output_dir:
        raise ValueError("No gcsOutputDirectory found in job status data.")
        
    print(f"Output directory in GCS: {gcs_output_dir}")
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.bucket(bucket_name.replace("gs://", ""))
    
    prefix = gcs_output_dir.replace(f"gs://{bucket.name}/", "").replace(f"gs://{bucket.name}", "")
    if prefix and not prefix.endswith("/"):
        prefix += "/"
        
    blobs = list(bucket.list_blobs(prefix=prefix))
    jsonl_blobs = [b for b in blobs if b.name.endswith(".jsonl")]
    
    if not jsonl_blobs:
        raise FileNotFoundError(f"No prediction JSONL files found in GCS prefix: {prefix}")
        
    print(f"Found {len(jsonl_blobs)} prediction JSONL files. Streaming and parsing...")
    
    parsed_data = []
    for blob in jsonl_blobs:
        print(f"  Streaming {blob.name} (Size: {blob.size / 1024 / 1024:.2f} MB) directly from GCS...")
        with blob.open("rt") as f:
            line_count = 0
            for line in f:
                if not line.strip():
                    continue
                parsed = extract_prediction_from_line(line)
                if parsed is None:
                    continue
                
                pred_id = parsed.get("id")
                pred_k = parsed.get("k")
                pred_class = parsed.get("class") or parsed.get("class_label") or "Noise"
                pred_explanation = parsed.get("explanation") or ""
                
                # Strip Class prefix
                if isinstance(pred_class, str):
                    pred_class = pred_class.strip()
                    if pred_class.lower().startswith("class "):
                        pred_class = pred_class[6:].strip()
                
                parsed_data.append({
                    "sclk": str(pred_id),
                    "predicted_label": pred_class,
                    "explanation": pred_explanation,
                    "k": int(pred_k) if pred_k is not None else 4
                })
                line_count += 1
                if line_count % 100 == 0:
                    print(f"    Streamed and parsed {line_count} predictions...")
                    
    predictions_df = pd.DataFrame(parsed_data)
    test_lookup = test_df[['sclk', 'class']].copy()
    test_lookup['sclk'] = test_lookup['sclk'].astype(str)
    
    merged_df = pd.merge(predictions_df, test_lookup, on='sclk', how='inner')
    merged_df.rename(columns={"class": "true_label"}, inplace=True)
    merged_df = merged_df[['sclk', 'true_label', 'predicted_label', 'explanation', 'k']]
    
    os.makedirs(os.path.dirname(output_parquet_path), exist_ok=True)
    merged_df.to_parquet(output_parquet_path)
    print(f"Saved results to {output_parquet_path}")
    return merged_df

print("\n--- STAGE 1: Multiclass Batch Predictions (Single Job) ---")
job_status = submit_and_poll_batch_job(configs, batch_file)

print("\nDownloading and parsing Stage 1 prediction results (streaming)...")
output_parquet = os.path.join(output_dir, "results_3p_ablation.parquet")
results_df = download_and_parse_batch_results_streaming(configs, job_status, test_df, output_parquet)

output_csv = os.path.join(output_dir, "results_3p_ablation.csv")
results_df[['sclk', 'true_label', 'predicted_label', 'explanation']].to_csv(output_csv, index=False)
print(f"Saved final predictions to CSV: {output_csv}")


## Step 8: Evaluate Multiclass Prediction Results & Plot Confusion Matrix Heatmap


In [ ]:
def generate_metrics_and_plots(results_df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    k_values = sorted(results_df['k'].unique())
    metrics_summary = {}
    
    print("\n" + "="*50 + "\n3-P focused Ablation Study Summary\n" + "="*50)
    
    for k in k_values:
        sub_df = results_df[results_df['k'] == k]
        
        # Overall accuracy
        overall_acc = accuracy_score(sub_df['true_label'].astype(str), sub_df['predicted_label'].astype(str))
        print(f"\nk = {k}-shot classification:")
        print(f"  Overall Accuracy: {overall_acc:.2%}")
        
        # Calculate confusion matrix for all classes
        labels = sorted(results_df['true_label'].astype(str).unique())
        cm = confusion_matrix(sub_df['true_label'].astype(str), sub_df['predicted_label'].astype(str), labels=labels)
        
        # Save Confusion Matrix as CSV
        cm_df = pd.DataFrame(cm, index=labels, columns=labels)
        cm_csv_path = os.path.join(output_dir, f"confusion_matrix_k{k}.csv")
        cm_df.to_csv(cm_csv_path)
        print(f"  Saved confusion matrix CSV to {cm_csv_path}")
        
        # Plot Confusion Matrix
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
        plt.title(f"Confusion Matrix (k={k} shots) - Acc: {overall_acc:.2%}")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.tight_layout()
        
        cm_path = os.path.join(output_dir, f"confusion_matrix_k{k}.png")
        plt.savefig(cm_path)
        plt.close()
        print(f"  Saved confusion matrix plot to {cm_path}")
        
        # Calculate class-specific metrics for 3-P
        target_cls = "3-P"
        
        # True Positives, False Positives, False Negatives, True Negatives
        y_true_binary = (sub_df['true_label'].astype(str) == target_cls).astype(int)
        y_pred_binary = (sub_df['predicted_label'].astype(str) == target_cls).astype(int)
        
        cm_binary = confusion_matrix(y_true_binary, y_pred_binary)
        if cm_binary.shape == (2, 2):
            tn, fp, fn, tp = cm_binary.ravel()
        else:
            tn, fp, fn, tp = 0, 0, 0, 0
            if len(y_true_binary.unique()) == 1:
                val = y_true_binary.iloc[0]
                if val == 0:
                    tn = len(y_true_binary)
                else:
                    tp = len(y_true_binary)
            
        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        
        print(f"  Class 3-P Specific Metrics:")
        print(f"    TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}")
        print(f"    Precision: {precision:.2%}")
        print(f"    Recall (Sensitivity): {recall:.2%}")
        print(f"    F1-score: {f1:.2%}")
        
        metrics_summary[str(k)] = {
            "overall_accuracy": float(overall_acc),
            "class_3p_metrics": {
                "tp": int(tp),
                "fp": int(fp),
                "fn": int(fn),
                "tn": int(tn),
                "precision": float(precision),
                "recall": float(recall),
                "f1_score": float(f1)
            }
        }
        
    metrics_json_path = os.path.join(output_dir, "metrics_3p_ablation_peaks_focused.json")
    with open(metrics_json_path, 'w') as f:
        json.dump(metrics_summary, f, indent=4)
    print(f"\nSaved metrics summary JSON to {metrics_json_path}")

generate_metrics_and_plots(results_df, output_dir)


## Step 9: Recheck Confusion Matrix from Saved CSV


In [ ]:
csv_path = os.path.join(output_dir, "results_3p_ablation.csv")
if os.path.exists(csv_path):
    print(f"Loading predictions from {csv_path}...")
    df_eval = pd.read_csv(csv_path)
    
    # Clean Class prefix
    def clean_label(label):
        if not isinstance(label, str):
            return "Noise"
        label = label.strip()
        if label.lower().startswith("class "):
            return label[6:].strip()
        return label
    
    df_eval['predicted_label'] = df_eval['predicted_label'].apply(clean_label)
    df_eval['true_label'] = df_eval['true_label'].astype(str)
    
    # Calculate overall metrics
    overall_acc = accuracy_score(df_eval['true_label'], df_eval['predicted_label'])
    print(f"Overall Accuracy from CSV: {overall_acc:.2%}")
    
    # Calculate confusion matrix
    labels = sorted(df_eval['true_label'].unique())
    cm = confusion_matrix(df_eval['true_label'], df_eval['predicted_label'], labels=labels)
    
    # Print Confusion Matrix Table
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    print("\nConfusion Matrix:")
    print(cm_df)
    
    # Plot Confusion Matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f"Cassini CDA Confusion Matrix (Acc: {overall_acc:.2%})")
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.tight_layout()
    plt.show()
    
    # Class-specific metrics for 3-P
    target_cls = "3-P"
    if target_cls in labels:
        y_true_binary = (df_eval['true_label'] == target_cls).astype(int)
        y_pred_binary = (df_eval['predicted_label'] == target_cls).astype(int)
        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        print(f"\nClass {target_cls} Metrics:")
        print(f"  Precision: {precision:.2%}")
        print(f"  Recall (Sensitivity): {recall:.2%}")
        print(f"  F1-score: {f1:.2%}")
else:
    print(f"File not found: {csv_path}")
